# Notebook 02: Smoke-Test der Eval-Metriken

**Forschungsarbeit:** Evolutionäre Hyperparameter-Optimierung für RAG-Pipelines

Dieses Notebook deckt folgende Schritte ab:

1. Eine Frage aus der Trainingsmenge laden
2. Beleg-Sätze über `load_sentences` aus dem Korpus einlesen
3. Drei synthetische `Document`-Sets bauen (Volltreffer, Teiltreffer, Fehltreffer)
4. Die vier LLM-freien Metriken (`context_recall`, `context_precision`, `answer_f1`, `exact_match`) pro Szenario berechnen und vergleichen
5. Den `composite_score` mit den Fitness-Gewichten aus der YAML-Config berechnen

Ziel ist die visuelle Bestätigung, dass die Metriken plausible Werte liefern, bevor die echte RAG-Pipeline ausgeführt wird.

## Setup

In [1]:
from src.environment import Environment
from src.evaluation.aggregate import print_metrics_table
from src.evaluation.fitness import composite_score
from src.evaluation.loader import load_qa_split, load_sentence_document, load_sentences
from src.evaluation.metrics import answer_f1, context_precision, context_recall, exact_match

env = Environment()

QA_SPLIT_PATH = env.qa_split_path
CORPUS_DIR = env.corpus_dir

print(env.summary())

experiment_name: ga_rag_optimizer_v1
seed: 42
raw_dir: data\raw
processed_dir: data\processed
corpus_dir: data\corpus
qa_split_path: data\processed\qa_pairs_split.json
corpus_meta_path: data\processed\corpus_meta.json
chroma_dir: data\processed\chroma_db
results_dir: results
logs_dir: results\logs
figures_dir: results\figures
tables_dir: results\tables


## Schritt 1: Eine HotpotQA-Frage laden

Der erste Eintrag aus der Trainingsmenge zeigt ein typisches Beispiel und dessen supporting_facts.

In [2]:
split = load_qa_split(QA_SPLIT_PATH)
qa = split["train"][0]

print(f"ID:      {qa.id}")
print(f"Typ:     {qa.type}")
print(f"Frage:   {qa.question}")
print(f"Antwort: {qa.answer}")
print(f"\nBelegende Sätze ({len(qa.supporting_facts)}):")
for fact in qa.supporting_facts:
    print(f"  - {fact.paragraph_id} (Satz {fact.sent_id})")
print(f"\nKontext: {len(qa.context_paragraph_ids)} Paragraphen")

ID:      5ae3df6e5542992e3233c48e
Typ:     bridge
Frage:   Indian Mound Cottage was owned by the financier that co-founded which company?
Antwort: Standard Oil

Belegende Sätze (3):
  - indian_mound_cottage (Satz 0)
  - william_rockefeller (Satz 0)
  - william_rockefeller (Satz 1)

Kontext: 10 Paragraphen


## Schritt 2: Beleg-Sätze aus dem Korpus einlesen

Die Korpus-Dateien haben den Titel in Zeile 1, dann eine Leerzeile, dann einen Satz pro Zeile. `load_sentences` liest die Sätze aus, daraus entstehen gleich die Chunk-Texte für die Szenarien.

In [3]:
first_fact = qa.supporting_facts[0]
sentences = load_sentences(CORPUS_DIR, first_fact.paragraph_id)
print(f"Paragraph '{first_fact.paragraph_id}', {len(sentences)} Sätze")
print(f"Belegender Satz {first_fact.sent_id}: {sentences[first_fact.sent_id][:200]}")

Paragraph 'indian_mound_cottage', 4 Sätze
Belegender Satz 0: Indian Mound Cottage is a mansion that was owned by oil executive William Rockefeller.


## Schritt 3: Drei synthetische Retrieval-Szenarien

Pro Szenario wird ein `Document`-Set gebaut, das ein realistisches Retrieval-Ergebnis nachbildet:

1. **Volltreffer**: alle Gold-Paragraphen exakt im Retrieval.
2. **Teiltreffer**: nur die Hälfte der Gold-Paragraphen, plus ein Distraktor.
3. **Fehltreffer**: drei Distraktor-Paragraphen, keine Gold-Treffer.

Jedes `Document` enthält einen echten Korpus-Satz und seine `paragraph_id` in den Metadaten, im selben Format wie später bei der echten Pipeline.

In [4]:
perfect_retrieved = []
for fact in qa.supporting_facts:
    doc = load_sentence_document(CORPUS_DIR, fact.paragraph_id, fact.sent_id)
    perfect_retrieved.append(doc)

gold_para_ids = set()
for fact in qa.supporting_facts:
    gold_para_ids.add(fact.paragraph_id)

distractor_ids = []
for para_id in qa.context_paragraph_ids:
    if para_id not in gold_para_ids:
        distractor_ids.append(para_id)

half = len(qa.supporting_facts) // 2
if half == 0:
    half = 1

partial_retrieved = []
for fact in qa.supporting_facts[:half]:
    doc = load_sentence_document(CORPUS_DIR, fact.paragraph_id, fact.sent_id)
    partial_retrieved.append(doc)
if distractor_ids:
    partial_retrieved.append(load_sentence_document(CORPUS_DIR, distractor_ids[0]))

miss_retrieved = []
for para_id in distractor_ids[:3]:
    miss_retrieved.append(load_sentence_document(CORPUS_DIR, para_id))

print(f"Volltreffer: {len(perfect_retrieved)} Chunks")
print(f"Teiltreffer: {len(partial_retrieved)} Chunks")
print(f"Fehltreffer: {len(miss_retrieved)} Chunks")

Volltreffer: 3 Chunks
Teiltreffer: 2 Chunks
Fehltreffer: 3 Chunks


## Schritt 4: Alle vier Metriken anwenden

Erwartung: Volltreffer liefert überall 1.0, Fehltreffer 0.0, Teiltreffer liegt dazwischen. `answer_f1` und `exact_match` hängen zusätzlich davon ab, ob die Antwort wörtlich in einem gefundenen Satz steht. Die Tabelle druckt `print_metrics_table` aus `src/evaluation/aggregate.py`.

In [5]:
scenarios = {
    "Volltreffer": perfect_retrieved,
    "Teiltreffer": partial_retrieved,
    "Fehltreffer": miss_retrieved,
}

scenario_metrics: dict[str, dict[str, float]] = {}
for name, retrieved in scenarios.items():
    scenario_metrics[name] = {
        "context_recall": context_recall(retrieved, qa.supporting_facts),
        "context_precision": context_precision(retrieved, qa.supporting_facts),
        "answer_f1": answer_f1(retrieved, qa.answer),
        "exact_match": exact_match(retrieved, qa.answer),
    }

print_metrics_table(scenario_metrics, row_label="Szenario")

Szenario      recall  precision  answer_f1  exact_match
-------------------------------------------------------
Volltreffer    1.000      1.000      1.000        1.000
Teiltreffer    0.500      0.500      0.500        0.000
Fehltreffer    0.000      0.000      0.000        0.000


## Schritt 5: Composite Score als GA-Fitness

Der GA maximiert nicht die Einzelmetriken, sondern ihre gewichtete Summe. Die Gewichte kommen aus `configs/experiment_config.yaml`: `context_recall` 0.4, `context_precision` 0.3, `answer_f1` 0.3. `exact_match` wird nur genannt und geht nicht in die Fitness ein. Erwartung: Volltreffer ergibt 1.0, Fehltreffer 0.0, Teiltreffer liegt dazwischen.

In [6]:
weights = env.config["evaluation"]["fitness_weights"]
print(f"Fitness-Gewichte: {weights}")

scores: dict[str, float] = {}
print(f"\n{'Szenario':<12} {'composite':>10}")
print("-" * 23)
for name, retrieved in scenarios.items():
    metrics = {
        "context_recall": context_recall(retrieved, qa.supporting_facts),
        "context_precision": context_precision(retrieved, qa.supporting_facts),
        "answer_f1": answer_f1(retrieved, qa.answer),
    }
    scores[name] = composite_score(metrics, weights)
    print(f"{name:<12} {scores[name]:>10.3f}")

assert scores["Volltreffer"] == 1.0, "Szenario Volltreffer sollte Composite 1.0 ergeben"
assert scores["Fehltreffer"] == 0.0, "Szenario Fehltreffer sollte Composite 0.0 ergeben"

Fitness-Gewichte: {'context_recall': 0.4, 'context_precision': 0.3, 'answer_f1': 0.3}

Szenario      composite
-----------------------
Volltreffer       1.000
Teiltreffer       0.500
Fehltreffer       0.000


## Plausibilitäts-Checks

Harte Erwartungen an die Extremfälle: Volltreffer muss Recall und Precision 1.0 liefern, Fehltreffer jeweils 0.0. Schlägt eine Prüfung fehl, liegt ein Bug in der Metrik oder im Szenariobau vor.

In [7]:
assert context_recall(perfect_retrieved, qa.supporting_facts) == 1.0, (
    "Szenario Volltreffer sollte Recall 1.0 ergeben"
)
assert context_precision(perfect_retrieved, qa.supporting_facts) == 1.0, (
    "Szenario Volltreffer sollte Precision 1.0 ergeben"
)
assert context_recall(miss_retrieved, qa.supporting_facts) == 0.0, (
    "Szenario Fehltreffer sollte Recall 0.0 ergeben"
)
assert context_precision(miss_retrieved, qa.supporting_facts) == 0.0, (
    "Szenario Fehltreffer sollte Precision 0.0 ergeben"
)

print("Alle Checks bestanden. Metriken zeigen die erwarteten Werte.")

Alle Checks bestanden. Metriken zeigen die erwarteten Werte.
